# Tugas 1 — 01: Data Profiling

**Dataset:** Wikipedia Bahasa Indonesia — dump *stub meta history* `idwiki-20260901` dari [Wikimedia dumps](https://dumps.wikimedia.org/idwiki/20260901/), dikonversi ke Parquet.

| Item | Nilai |
|---|---|
| Sumber | `https://dumps.wikimedia.org/idwiki/20260901/` |
| Data mentah | `data/raw/idwiki-20260901-stub-meta-history1.xml.gz` (2,34 GB, MD5 terverifikasi) |
| Data olahan | `data/processed/idwiki_revisions.parquet` (1,96 GB) |
| Jumlah baris | 27.771.250 revisi (posisi 1 September 2026) |
| Unit analisis | 1 baris = 1 revisi (edit) pada 1 halaman |

Notebook ini memprofil dataset dengan **Polars** menggunakan *lazy evaluation* (`scan_parquet`): skema, dimensi, missing values, nilai unik, statistik deskriptif, cakupan waktu, dan distribusi kolom kategorikal.

> Konversi XML → Parquet dilakukan oleh `src/convert_idwiki_to_parquet.py` dan dapat dijalankan ulang kapan saja.

## Setup

Variabel `DATA_PATH` menunjuk ke file dataset yang dipakai seluruh notebook (sesuai petunjuk `README.md`).

In [1]:
from pathlib import Path

import polars as pl

DATA_PATH = Path('data/processed/idwiki_revisions.parquet')
if not DATA_PATH.exists():  # fallback bila kernel berjalan dari folder notebooks/
    DATA_PATH = Path('..') / DATA_PATH
assert DATA_PATH.exists(), f'File tidak ditemukan: {DATA_PATH.resolve()}'

lf = pl.scan_parquet(DATA_PATH)  # lazy: data belum dibaca ke memori
print('DATA_PATH :', DATA_PATH)
print('ukuran    :', f'{DATA_PATH.stat().st_size / 1e9:.2f} GB')

DATA_PATH : ../data/processed/idwiki_revisions.parquet
ukuran    : 1.96 GB


## 1. Skema dan dimensi

Tipe data per kolom, jumlah baris, dan cuplikan 10 baris pertama.

In [2]:
schema = lf.collect_schema()
for nama, tipe in schema.items():
    print(f'{nama:20s} {tipe}')

page_id              Int64
page_title           String
page_namespace       Int16
is_redirect          Boolean
rev_id               Int64
parent_rev_id        Int64
rev_timestamp        Datetime(time_unit='us', time_zone=None)
contributor_name     String
contributor_id       Int64
is_anonymous         Boolean
is_minor             Boolean
comment              String
text_bytes           Int64
text_sha1            String
rev_sha1             String


In [3]:
total_baris = lf.select(pl.len()).collect().item()
print(f'Jumlah baris : {total_baris:,}')
print(f'Jumlah kolom : {len(schema)}')

lf.head(10).collect()

Jumlah baris : 27,771,250
Jumlah kolom : 15


page_id,page_title,page_namespace,is_redirect,rev_id,parent_rev_id,rev_timestamp,contributor_name,contributor_id,is_anonymous,is_minor,comment,text_bytes,text_sha1,rev_sha1
i64,str,i16,bool,i64,i64,datetime[μs],str,i64,bool,bool,str,i64,str,str
1,"""Asam deoksiribonukleat""",0,false,1,null,2003-11-07 00:43:23,"""proxy8.ntu.edu.sg""",0,false,true,"""*""",27,"""8z86w02pi069jvecmz9wxy0rlibfnj…","""8z86w02pi069jvecmz9wxy0rlibfnj…"
1,"""Asam deoksiribonukleat""",0,false,2,1,2003-11-07 00:43:35,"""proxy8.ntu.edu.sg""",0,false,false,"""*""",31,"""1fjnvhloxoz7o9781dofogu5zoczxe…","""1fjnvhloxoz7o9781dofogu5zoczxe…"
1,"""Asam deoksiribonukleat""",0,false,3,2,2003-11-07 00:52:09,"""155.69.5.xxx""",0,false,false,"""*""",296,"""c3dsimxeex3lj1f85144xjybzc0qtc…","""c3dsimxeex3lj1f85144xjybzc0qtc…"
1,"""Asam deoksiribonukleat""",0,false,4,3,2003-11-07 01:04:46,"""155.69.5.xxx""",0,false,false,"""*""",639,"""hcxzgjul5kvci4vlgjkb3khkrblndq…","""hcxzgjul5kvci4vlgjkb3khkrblndq…"
1,"""Asam deoksiribonukleat""",0,false,5,4,2003-11-07 01:43:34,"""155.69.5.xxx""",0,false,false,"""*""",2281,"""2dff5now9prer3zn7ztx8q44r92c1x…","""2dff5now9prer3zn7ztx8q44r92c1x…"
1,"""Asam deoksiribonukleat""",0,false,6,5,2003-11-07 09:39:07,"""Vyasa""",756234,false,false,"""*Tambahkan 'link' ke definisi …",2871,"""sud0v2s6lgl03zh3qywbjglktrkqn4…","""sud0v2s6lgl03zh3qywbjglktrkqn4…"
1,"""Asam deoksiribonukleat""",0,false,7,6,2003-11-07 10:20:52,"""Vyasa""",756234,false,false,"""*Tambahkan gambar DNA""",3163,"""1px7evdmxakwc6t2oz7tnjuepermdk…","""1px7evdmxakwc6t2oz7tnjuepermdk…"
1,"""Asam deoksiribonukleat""",0,false,86,7,2003-11-07 12:12:58,"""Vyasa""",756234,false,false,"""*Perbaiki garis bawah (ubah ""<…",3161,"""22x4u5f5dfyrovq8fgkjv36uzglfhz…","""22x4u5f5dfyrovq8fgkjv36uzglfhz…"
1,"""Asam deoksiribonukleat""",0,false,155,86,2004-01-29 21:39:08,"""Andre Engels""",4,false,true,null,2794,"""9woccwlqbrmfmey68k4zzp0budwpih…","""9woccwlqbrmfmey68k4zzp0budwpih…"


## 2. Missing values

Jumlah dan persentase null pada setiap kolom terhadap total baris.

> Dihitung **per kolom secara berurutan** — memilih 15 kolom sekaligus akan menarik seluruh kolom teks ke memori dan memicu OOM pada container ±4 GB.

In [4]:
baris_per_kolom = {}
for k in schema:
    baris_per_kolom[k] = lf.select(pl.col(k).is_null().sum()).collect().item()
nulls = pl.DataFrame(
    {'kolom': list(baris_per_kolom), 'jumlah_null': list(baris_per_kolom.values())}
)
nulls = nulls.with_columns(
    (pl.col('jumlah_null') / total_baris * 100).round(3).alias('persen_null')
).sort('jumlah_null', descending=True)
nulls

kolom,jumlah_null,persen_null
str,i64,f64
"""comment""",5252300,18.913
"""parent_rev_id""",4255798,15.324
"""contributor_id""",2199136,7.919
"""rev_sha1""",8382,0.03
"""contributor_name""",5571,0.02
…,…,…
"""rev_timestamp""",0,0.0
"""is_anonymous""",0,0.0
"""is_minor""",0,0.0


## 3. Nilai unik

Jumlah nilai unik (dikurangi null) pada kolom identitas — untuk melihat skala entitas dan mendeteksi potensi duplikasi.

> Sama seperti sebelumnya: satu kolom per eksekusi agar puncak memori tetap aman.

In [5]:
kolom_unik = ['page_id', 'page_title', 'rev_id', 'parent_rev_id', 'contributor_name']
nilai_unik = {
    k: lf.select(pl.col(k).drop_nulls().n_unique()).collect().item()
    for k in kolom_unik
}
pl.DataFrame({'kolom': list(nilai_unik), 'nilai_unik': list(nilai_unik.values())})

kolom,nilai_unik
str,i64
"""page_id""",4253661
"""page_title""",4253661
"""rev_id""",27771250
"""parent_rev_id""",23515014
"""contributor_name""",966298


## 4. Statistik deskriptif (kolom numerik)

Agregasi via lazy expressions — seluruh 27,7 juta baris tidak pernah dimuat penuh ke memori; tiap kolom dihitung terpisah (min, kuartil, mean, std).

In [6]:
kolom_num = ['page_namespace', 'rev_id', 'parent_rev_id', 'text_bytes']
fungsi_stat = {
    'min': lambda k: pl.col(k).min(),
    'q25': lambda k: pl.col(k).quantile(0.25),
    'median': lambda k: pl.col(k).median(),
    'mean': lambda k: pl.col(k).mean(),
    'q75': lambda k: pl.col(k).quantile(0.75),
    'max': lambda k: pl.col(k).max(),
    'std': lambda k: pl.col(k).std(),
}
baris = []
for k in kolom_num:
    nilai = lf.select([fn(k).alias(nama) for nama, fn in fungsi_stat.items()]).collect().row(0)
    baris.append({'kolom': k, **dict(zip(fungsi_stat, nilai))})
pl.DataFrame(baris)

kolom,min,q25,median,mean,q75,max,std
str,i64,f64,f64,f64,f64,i64,f64
"""page_namespace""",0,0.0,0.0,2.873131,1.0,2600,47.361903
"""rev_id""",1,7.485626e6,1.4960e7,1.4947e7,2.2460383e7,29684531,8.5885e6
"""parent_rev_id""",1,6.966883e6,1.4388e7,1.4344e7,2.1386183e7,29684494,8.4351e6
"""text_bytes""",0,980.0,2516.0,9251.012579,6770.0,2096646,28425.480558


## 5. Cakupan waktu

Rentang timestamp revisi dan jumlah revisi per tahun — dasar analisis temporal pada Milestone 3.

In [7]:
lf.select(
    pl.col('rev_timestamp').min().alias('revisi_pertama'),
    pl.col('rev_timestamp').max().alias('revisi_terakhir'),
).collect()

revisi_pertama,revisi_terakhir
datetime[μs],datetime[μs]
2003-01-24 16:06:03,2026-09-01 12:57:27


In [8]:
per_tahun = (
    lf.group_by(pl.col('rev_timestamp').dt.year().alias('tahun'))
    .agg(pl.len().alias('jumlah_revisi'))
    .sort('tahun')
    .collect()
)
per_tahun

tahun,jumlah_revisi
i32,u32
2003,130
2004,15175
2005,94225
2006,333513
2007,580385
…,…
2022,2641295
2023,2438911
2024,1548548


## 6. Distribusi kolom kategorikal

Sebaran revisi per namespace halaman dan proporsi revisi anonim/minor/redirect.

In [9]:
per_ns = (
    lf.group_by('page_namespace')
    .agg(pl.len().alias('jumlah_revisi'))
    .sort('jumlah_revisi', descending=True)
    .collect()
)
per_ns.head(10)

page_namespace,jumlah_revisi
i16,u32
0,20358320
3,2181462
1,2054374
14,1061123
10,650851
4,502038
2,501678
6,222346
15,74132


In [10]:
flags = lf.select(
    pl.len().alias('total_revisi'),
    pl.col('is_anonymous').sum().alias('revisi_anonim'),
    pl.col('is_minor').sum().alias('revisi_minor'),
    pl.col('is_redirect').sum().alias('revisi_di_halaman_redirect'),
).collect()
flags.with_columns(
    (pl.col('revisi_anonim') / pl.col('total_revisi') * 100).round(2).alias('pct_anonim'),
    (pl.col('revisi_minor') / pl.col('total_revisi') * 100).round(2).alias('pct_minor'),
    (pl.col('revisi_di_halaman_redirect') / pl.col('total_revisi') * 100).round(2).alias('pct_redirect'),
)

total_revisi,revisi_anonim,revisi_minor,revisi_di_halaman_redirect,pct_anonim,pct_minor,pct_redirect
u32,u32,u32,u32,f64,f64,f64
27771250,2193565,12795917,1422420,7.9,46.08,5.12


## Ringkasan temuan profiling

- Dataset berisi **27.771.250 baris** × **15 kolom** dari **4.253.661 halaman** unik — jauh melampaui syarat tugas (>1 juta baris atau ≥500 MB).
- Cakupan waktu **24 Januari 2003 – 1 September 2026**; volume revisi menumpuk pada tahun-tahun aktif (lihat tabel revisi per tahun).
- `rev_id` unik = jumlah baris (tidak ada duplikasi ID revisi); `page_id` dan `page_title` sama-sama 4.253.661 unik; **966.298 kontributor unik**.
- Null terbesar: `comment` 18,9% (suntingan tanpa ringkasan), `parent_rev_id` 15,3% — hampir persis jumlah halaman, sesuai harapan karena revisi pertama tiap halaman tidak punya parent — dan `contributor_id` 7,9% (kontributor IP/anonim). Kolom kunci (`page_id`, `page_title`, `rev_id`, `rev_timestamp`) praktis bebas null.
- `text_bytes` sangat miring ke kanan: median 2.516 B, mean 9.251 B, maksimum ±2,1 juta B — kandidat outlier yang ditangani pada tahap cleaning.

**Lanjutan:** `02_data_cleaning.ipynb` (Milestone 2) — penanganan null, duplikasi, outlier, dan validasi kualitas dengan Polars, dilengkapi profiling DuckDB `SUMMARIZE`.